# Clinical Trial Simulator - Resume-First GRPO Training (Colab)

This notebook is designed to **continue from completed execution**. It reads existing training artifacts first, plots real evidence, and only then optionally resumes GRPO training.

Submission focus:
- Novel environment: multi-objective clinical trial control (safety, efficacy, regulation, budget, supply)
- Real evidence: reward and loss curves from saved trainer state + benchmark comparisons
- Story: what changed after training and why it matters

In [ ]:
#@title 1) Setup
import os
import sys
import json
import subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
WORKDIR = Path('/content/Clinical-Trial-Simulator') if IN_COLAB else Path.cwd()

if IN_COLAB:
    if not WORKDIR.exists():
        subprocess.run(['git', 'clone', 'https://github.com/TheSun-1712/Clinical-Trial-Simulator.git', str(WORKDIR)], check=True)
    os.chdir(WORKDIR)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[train]'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'matplotlib', 'seaborn'], check=True)
print('Working directory:', Path.cwd())

In [ ]:
#@title 2) Locate completed artifacts
from pathlib import Path

ROOT = Path.cwd()
ART = ROOT / 'artifacts'
POLICY_MANIFEST = ART / 'policy' / 'latest_llm.json'
BENCH_SUMMARY = ART / 'benchmark' / 'latest_summary.json'
BENCH_TIMELINE = ART / 'benchmark' / 'latest_timeline.json'
TRAINER_STATE = ART / 'trl_gpu_8gb' / 'checkpoint-50' / 'trainer_state.json'
PLOTS = ART / 'plots'
PLOTS.mkdir(parents=True, exist_ok=True)

required = [POLICY_MANIFEST, BENCH_SUMMARY, BENCH_TIMELINE, TRAINER_STATE]
missing = [str(p) for p in required if not p.exists()]
if missing:
    print('Missing artifacts:')
    for m in missing:
        print(' -', m)
    print('You can still run training below, but this notebook is optimized for resume-first evidence.')
else:
    print('All resume artifacts found.')

In [ ]:
#@title 3) Read completed run metrics
import json

if POLICY_MANIFEST.exists():
    policy = json.loads(POLICY_MANIFEST.read_text(encoding='utf-8'))
    meta = policy.get('metadata', {})
    print('Model:', policy.get('model_name'))
    print('Reward mean (training):', meta.get('reward_mean'))
    print('Valid JSON rate:', meta.get('valid_json_rate'))
    print('Valid action rate:', meta.get('valid_action_rate'))
    print('Reward mode:', meta.get('reward_mode'))
    print('Eval summary:', meta.get('eval_summary'))

if BENCH_SUMMARY.exists():
    bench = json.loads(BENCH_SUMMARY.read_text(encoding='utf-8'))
    print('\nPolicy rows from benchmark:')
    for row in bench.get('policy_rows', []):
        print(row['policy'], 'total_reward=', round(row['total_reward'], 4), 'source=', row.get('source', ''))

In [ ]:
#@title 4) Generate evidence plots from completed artifacts
import json
import pandas as pd
import matplotlib.pyplot as plt

trainer = json.loads(TRAINER_STATE.read_text(encoding='utf-8'))
history = trainer.get('log_history', [])
hdf = pd.DataFrame(history)

fig, ax1 = plt.subplots(figsize=(10, 5.5))
ax1.plot(hdf['step'], hdf['reward'], marker='o', linewidth=2.2, color='#0072B2', label='GRPO reward')
ax1.set_xlabel('Training step')
ax1.set_ylabel('Reward (mean)', color='#0072B2')
ax1.tick_params(axis='y', labelcolor='#0072B2')
ax1.grid(alpha=0.25)

ax2 = ax1.twinx()
ax2.plot(hdf['step'], hdf['loss'], marker='s', linestyle='--', linewidth=1.8, color='#D55E00', label='Policy loss')
ax2.set_ylabel('Loss', color='#D55E00')
ax2.tick_params(axis='y', labelcolor='#D55E00')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='lower right')
ax1.set_title('GRPO Training Dynamics (resumed run)')
fig.tight_layout()
reward_loss_path = PLOTS / 'reward_loss_curves.png'
fig.savefig(reward_loss_path, dpi=160)
plt.show()

bench = json.loads(BENCH_SUMMARY.read_text(encoding='utf-8'))
bdf = pd.DataFrame(bench['policy_rows'])
fig, ax = plt.subplots(figsize=(9, 5.2))
ax.bar(bdf['policy'], bdf['total_reward'], color=['#999999', '#E69F00', '#009E73'])
ax.axhline(0.0, color='#333333', linewidth=1)
ax.set_xlabel('Policy')
ax.set_ylabel('Mean episode total reward')
ax.set_title('Benchmark: random vs heuristic vs trained')
for i, v in enumerate(bdf['total_reward']):
    ax.text(i, v + (0.2 if v >= 0 else -0.35), f'{v:.2f}', ha='center', va='bottom' if v >= 0 else 'top')
ax.grid(axis='y', alpha=0.25)
fig.tight_layout()
policy_cmp_path = PLOTS / 'policy_comparison.png'
fig.savefig(policy_cmp_path, dpi=160)
plt.show()

timeline = pd.DataFrame(json.loads(BENCH_TIMELINE.read_text(encoding='utf-8')))
wdf = timeline.groupby(['policy', 'week'], as_index=False)['total_reward'].mean().sort_values(['policy', 'week'])
fig, ax = plt.subplots(figsize=(10, 5.5))
palette = {'random': '#888888', 'heuristic': '#CC79A7', 'trained': '#009E73'}
for policy_name, sdf in wdf.groupby('policy'):
    ax.plot(sdf['week'], sdf['total_reward'], linewidth=2, label=policy_name, color=palette.get(policy_name))
ax.set_xlabel('Week')
ax.set_ylabel('Per-step reward')
ax.set_title('Weekly reward trajectory')
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
timeline_path = PLOTS / 'weekly_reward_timeline.png'
fig.savefig(timeline_path, dpi=160)
plt.show()

print('Saved plots:')
print('-', reward_loss_path)
print('-', policy_cmp_path)
print('-', timeline_path)

## Story Snapshot (for README/blog/video)

In this run, the trained policy beats the random and heuristic baselines on total reward, mainly by becoming a conservative, compliance-preserving controller under uncertainty.

The next frontier is exploration shaping: improve efficacy while preserving that safety/compliance behavior.

In [ ]:
#@title 5) Optional: continue training from completed checkpoint (do not restart)
# Set CONTINUE_TRAINING=True to resume; default is False to avoid accidental reruns.
CONTINUE_TRAINING = False

if CONTINUE_TRAINING:
    cmd = [
        sys.executable, 'training/train_grpo.py',
        '--backend', 'trl',
        '--config', 'training/configs/grpo_gpu_8gb.yaml',
        '--resume-from', str(POLICY_MANIFEST),
        '--max-steps', '200',
        '--output', 'artifacts/policy/latest_llm.json'
    ]
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True)
else:
    print('Skipped. Set CONTINUE_TRAINING=True to resume from existing policy manifest.')

In [ ]:
#@title 6) Optional: rerun benchmark after continuation
RERUN_BENCHMARK = False

if RERUN_BENCHMARK:
    cmd = [
        sys.executable, '-m', 'eval.run_benchmark',
        '--episodes', '12',
        '--trained-checkpoint', 'artifacts/policy/latest_llm.json',
        '--output-dir', 'artifacts/benchmark'
    ]
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True)
else:
    print('Skipped. Set RERUN_BENCHMARK=True when you want a fresh post-training comparison.')

## Submission Checklist

1. Keep the three generated plots under artifacts/plots and embed them in README.
2. Link this notebook in README using a Colab badge.
3. Add your Hugging Face Space URL, blog/video/slide URL(s), and any W&B run URL to README.
4. Push to a public HF Space and verify reset/step endpoints are live.
5. Ensure openenv.yaml is present at repo root and consistent with implemented API.